In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

In [4]:
# load training and test datasets
train = pd.read_csv("../Dataset/train.csv")
test  = pd.read_csv("../Dataset/test.csv")

print(f"Train Shape: {train.shape}")
print(f"Test Shape:  {test.shape}")

Train Shape: (630000, 21)
Test Shape:  (270000, 20)


In [5]:
# check column types to identify numeric vs categorical columns
train.dtypes

id                           int64
Soil_Type                   object
Soil_pH                    float64
Soil_Moisture              float64
Organic_Carbon             float64
Electrical_Conductivity    float64
Temperature_C              float64
Humidity                   float64
Rainfall_mm                float64
Sunlight_Hours             float64
Wind_Speed_kmh             float64
Crop_Type                   object
Crop_Growth_Stage           object
Season                      object
Irrigation_Type             object
Water_Source                object
Field_Area_hectare         float64
Mulching_Used               object
Previous_Irrigation_mm     float64
Region                      object
Irrigation_Need             object
dtype: object

In [ ]:
# check class balance — critical because we are scored on balanced accuracy
print(train['Irrigation_Need'].value_counts())
print()
print(train['Irrigation_Need'].value_counts(normalize=True))

In [ ]:
# check for missing values across all columns
missing = train.isnull().sum().sort_values(ascending=False)
print(missing)

In [ ]:
# automatically detect categorical columns instead of hardcoding them
# any column with dtype 'object' is categorical
# we exclude the target column 'Irrigation_Need' since we handle that separately
categorical_cols = train.select_dtypes(include='object').columns.tolist()
categorical_cols.remove('Irrigation_Need')

print("Categorical columns found:", categorical_cols)
print()

# show unique values for each categorical column
for col in categorical_cols:
    print(f"{col}: {train[col].nunique()} unique values → {train[col].unique()}")

In [ ]:
# ── Encode Target ──────────────────────────────────────────────────────────
# keep le_target separate so we can reverse predictions back to words later
le_target = LabelEncoder()
train['Irrigation_Need_encoded'] = le_target.fit_transform(train['Irrigation_Need'])

# print mapping so we never forget which number maps to which class
print("Target mapping:", dict(zip(le_target.classes_, le_target.transform(le_target.classes_))))

# ── Encode Categorical Features ────────────────────────────────────────────
# fit on train only to avoid data leakage
# transform applies the learned mapping without re-learning
le = LabelEncoder()
for col in categorical_cols:
    train[col] = le.fit_transform(train[col])  # learn from train and convert
    test[col]  = le.transform(test[col])        # only convert, never re-learn

print("\nEncoding done!")

In [ ]:
# visually confirm columns now contain numbers instead of text
train.head()